# GPT G6 — GPU idle diagnosis

Diagnostic only: frozen v4 solution, unchanged official workload and launch watcher. Three traced steps (11–13), CPU + CUDA with Python stacks, followed by a no-stack control. No performance score is claimed from profiled runs.

In [ ]:
import os,sys,subprocess,pathlib,json,hashlib,shutil,importlib.metadata as md
COMMIT="7ebed815c9cf3bbddc510474a6298b3254128802"
REPO=pathlib.Path("/kaggle/working/repo")
packages=["soup-cli[train]==0.77.0","transformers==5.16.1","peft==0.20.0","trl==0.29.1","bitsandbytes==0.50.2","accelerate==1.14.0","pytest"]
subprocess.run([sys.executable,"-m","pip","install","-q",*packages],check=True)
assert md.version("torch").startswith("2.11.0") and md.version("triton")=="3.6.0"
assert not {k:v for k,v in os.environ.items() if k.startswith("SKC_")}
print(subprocess.check_output(["nvidia-smi"],text=True))
shutil.rmtree(REPO,ignore_errors=True)
subprocess.run(["git","clone","-q","https://github.com/tokha-t/decik.git",str(REPO)],check=True)
subprocess.run(["git","checkout","--detach","-q",COMMIT],cwd=REPO,check=True)
subprocess.run(["git","diff","--exit-code","5c1bbe560c08b0323eefeb843a0baf00e400e774","--","solution","harness","tests","task","templates","LICENSE","NOTICE"],cwd=REPO,check=True)
subprocess.run([sys.executable,"task/prepare_data.py"],cwd=REPO,check=True)
OUT=pathlib.Path("/kaggle/working/g6-evidence");OUT.mkdir(exist_ok=True)
print("G6_SOURCE",COMMIT,flush=True)


In [ ]:
for tag,extra in [("with-stack",[]),("no-stack",["--no-stack"])]:
    run_dir=REPO/"trace_runs"/tag
    proc=subprocess.run([sys.executable,"dev/trace_run.py","--solution","--out",str(run_dir),*extra],cwd=REPO)
    if (run_dir/"train.log").exists():
        print((run_dir/"train.log").read_text()[-6000:],flush=True)
    proc.check_returncode()
    target=OUT/tag;target.mkdir(exist_ok=True)
    for name in ("trace.json.gz","metrics.json","run.json","train.log"):
        shutil.copy2(run_dir/name,target/name)
    metrics=json.loads((target/"metrics.json").read_text())
    assert len(metrics["steps"])==64 and metrics["trace_exported"]
    proc=subprocess.run([sys.executable,"dev/analyze_trace.py",str(target/"trace.json.gz"),"--out",str(target/"analysis.json")],cwd=REPO,capture_output=True,text=True)
    (target/"analysis.log").write_text(proc.stdout+"\n"+proc.stderr)
    print(proc.stdout,proc.stderr,flush=True)
    proc.check_returncode()
    print("G6_RESULT",tag,json.dumps({k:v for k,v in json.loads((target/"analysis.json").read_text()).items() if k not in ("gaps",)}),flush=True)
manifest={"source_commit":COMMIT,"frozen_solution":"5c1bbe560c08b0323eefeb843a0baf00e400e774","steps":[11,12,13],"client_date":"2026-10-31","client_timezone":"Asia/Almaty","versions":{x:md.version(x) for x in ("torch","triton","soup-cli","transformers","peft","trl","bitsandbytes","accelerate")},"sha256":{str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.rglob("*") if p.is_file()}}
(OUT/"manifest.json").write_text(json.dumps(manifest,indent=2))
shutil.make_archive("/kaggle/working/g6-evidence","zip",OUT)
print("G6_COMPLETE",json.dumps(manifest),flush=True)
